# Lyra 0.1 ? Kaggle GPU training

Run cells from top to bottom. Full model training requires an enabled Kaggle GPU. Checkpoints, tokenizer, data manifests, logs and exports are written under `/kaggle/working/Lyra0.1`; save a Notebook version to preserve them. Set `AUTO_RESUME=False` to start a fresh run directory.


## 1. Environment

Edit the run settings here. For a later session, attach this notebook's previous output from **Input ? Add Input ? Notebook Output Files**. The notebook finds compatible `latest.pt` files among attached inputs. Set `ALLOW_DATASET_CHANGE=True` when intentionally expanding/rebuilding data; model, config and tokenizer checks remain strict.


In [ ]:
from pathlib import Path
import os, sys, json, glob, shutil, subprocess, hashlib, statistics

REPO_URL = "https://github.com/er1kosfunpay-sketch/Lyra0.1.git"
PROJECT_DIR = Path("/kaggle/working/Lyra0.1")
RUN_NAME = "stage1_1300"  # use a new name for a distinct run; never point at an older run by accident
AUTO_RESUME = True
ALLOW_DATASET_CHANGE = True
USE_ATTACHED_DATA = True
RESUME_CHECKPOINT = None  # optionally set an exact /kaggle/input/.../latest.pt
TRAIN_STEPS = 2000
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 3e-4
WARMUP_STEPS = 100
SAVE_EVERY = 100
EVAL_EVERY = 250
KEEP_LAST_N_CHECKPOINTS = 3
CONFIG = "configs/kaggle.json"
OUTPUT_DIR = PROJECT_DIR / "checkpoints" / RUN_NAME
print({"project": str(PROJECT_DIR), "run": RUN_NAME, "auto_resume": AUTO_RESUME, "dataset_change_allowed": ALLOW_DATASET_CHANGE})


## 2. GPU check

Lyra 0.1 at sequence length 1024 should use a GPU with at least 12 GiB of total VRAM; Kaggle T4 x2 appears as two devices, but this stable pipeline trains on `cuda:0` only. It does not distribute work across both cards.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Notebook Settings select Accelerator ? GPU, then rerun.")
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB")
props = torch.cuda.get_device_properties(0)
if props.total_memory < 12 * 1024**3:
    raise RuntimeError(f"GPU 0 has {props.total_memory / 1024**3:.1f} GiB; Lyra context=1024 needs a larger GPU profile.")
x = torch.randn(1024, 1024, device="cuda:0"); y = x @ x
print("CUDA matmul:", float(y.mean()), "BF16 supported:", torch.cuda.is_bf16_supported())
del x, y; torch.cuda.empty_cache()


## 3. Clone or update repository


In [ ]:
if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone over non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("Repository:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 4. Install dependencies


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[data,dev]"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


## 5. Check project


In [ ]:
subprocess.run([sys.executable, "-m", "py_compile", "scripts/train.py", "lyra/checkpoint.py", "lyra/data.py"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", CONFIG, "--require-cuda", "--sanity"], cwd=PROJECT_DIR, check=True)


## 6. Check or prepare dataset

An attached Kaggle Dataset / prior Notebook output containing `train.jsonl` and `validation.jsonl` is reused. Otherwise the repository's existing builder prepares the initial OASST + Russian Everyday + curated corpus with its reviewed filters and splits. This can take time and needs Internet enabled.


In [ ]:
data_root = PROJECT_DIR / "data/processed/kaggle_stage1"
attached_pairs = []
for train_file in glob.glob("/kaggle/input/**/train.jsonl", recursive=True):
    val_file = Path(train_file).with_name("validation.jsonl")
    if val_file.is_file(): attached_pairs.append((Path(train_file), val_file))
if attached_pairs and USE_ATTACHED_DATA:
    src_train, src_val = attached_pairs[0]
    data_root.mkdir(parents=True, exist_ok=True)
    if src_train.resolve() != (data_root / "train.jsonl").resolve(): shutil.copy2(src_train, data_root / "train.jsonl")
    if src_val.resolve() != (data_root / "validation.jsonl").resolve(): shutil.copy2(src_val, data_root / "validation.jsonl")
    print("Using attached dataset:", src_train.parent)
elif not (data_root / "train.jsonl").is_file() or not (data_root / "validation.jsonl").is_file():
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--sources", "oasst,ru_everyday", "--max-per-source", "50000", "--balance", "fixed", "--ru-share", "0.5", "--out", str(data_root)], cwd=PROJECT_DIR, check=True)
train_file, val_file = data_root / "train.jsonl", data_root / "validation.jsonl"
if not train_file.is_file() or not val_file.is_file(): raise FileNotFoundError("Required train.jsonl / validation.jsonl missing. Attach a Kaggle dataset or allow dataset preparation.")


## 7. Check tokenizer

Reuse an attached 16,384-token Lyra tokenizer if present; otherwise train it only on the training split. A debug tokenizer is a hard stop.


In [ ]:
tokenizer_path = PROJECT_DIR / "artifacts/tokenizer/tokenizer.json"
attached_tok = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
if not tokenizer_path.is_file() and attached_tok:
    tokenizer_path.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(attached_tok[0], tokenizer_path)
if not tokenizer_path.is_file():
    subprocess.run([sys.executable, "scripts/train_tokenizer.py", "--data", str(train_file), "--vocab-size", "16384", "--out", str(tokenizer_path)], cwd=PROJECT_DIR, check=True)
from lyra.tokenizer import LyraTokenizer
from lyra.config import LyraConfig
from lyra.model import LyraModel
cfg = LyraConfig.from_json(CONFIG)
tok = LyraTokenizer.from_file(tokenizer_path)
expected_special = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SYSTEM>", "<USER>", "<ASSISTANT>", "<TOOL>", "<END>"]
if tok.vocab_size != 16384:
    raise RuntimeError(f"ERROR: Debug tokenizer detected. Expected vocab_size=16384; got {tok.vocab_size}.")
for special in expected_special: tok.id(special)
print("Tokenizer OK; vocab_size=", tok.vocab_size, "special_tokens=", expected_special)
tokenizer_fingerprint = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()


## 8. Find checkpoint

Attached Notebook outputs are read-only inputs. The trainer writes a new checkpoint set into this run's output folder and does not modify the attached source.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_candidates = sorted(glob.glob("/kaggle/input/**/latest.pt", recursive=True), key=lambda p: Path(p).stat().st_mtime, reverse=True)
resume_path = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
local_latest = OUTPUT_DIR / "latest.pt"
if AUTO_RESUME and resume_path is None:
    if local_latest.is_file(): resume_path = local_latest
    elif len(checkpoint_candidates) == 1: resume_path = Path(checkpoint_candidates[0])
    elif len(checkpoint_candidates) > 1:
        raise RuntimeError("Multiple attached latest.pt checkpoints found; set RESUME_CHECKPOINT to the intended run:\n" + "\n".join(checkpoint_candidates))
print("Resume checkpoint:", str(resume_path) if resume_path else "none; fresh run")
if resume_path:
    prior_log = resume_path.parent / "training.jsonl"
    if prior_log.is_file() and not (OUTPUT_DIR / "training.jsonl").exists(): shutil.copy2(prior_log, OUTPUT_DIR / "training.jsonl")


## 9. Configure and report training


In [ ]:
def inspect_jsonl(path):
    count = messages = chars = 0; langs = {}; turn_counts = []; approx_tokens = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line); count += 1
            ms = row.get("messages", []); messages += len(ms); turn_counts.append(len(ms))
            langs[row.get("lang", "unknown")] = langs.get(row.get("lang", "unknown"), 0) + 1
            for m in ms: chars += len(m.get("content", "")); approx_tokens += len(tok.encode(m.get("content", "")))
    return {"examples":count,"messages":messages,"languages":langs,"average_turns":round(statistics.mean(turn_counts),2) if turn_counts else 0,"approx_tokens":approx_tokens,"average_conversation_chars":round(chars/max(count,1),1)}
train_stats, val_stats = inspect_jsonl(train_file), inspect_jsonl(val_file)
model = LyraModel(cfg)
parameter_count = model.parameter_count()
del model
print("""========================
LYRA TRAINING
========================""")
print(json.dumps({"model_parameters":parameter_count,"tokenizer":str(tokenizer_path),"vocabulary":tok.vocab_size,"context":cfg.context_length,"dataset":str(data_root),"train":train_stats,"validation":val_stats,"gpu":torch.cuda.get_device_name(0),"gpu_count":torch.cuda.device_count(),"vram_gib":round(props.total_memory/1024**3,2),"dtype":"bfloat16" if torch.cuda.is_bf16_supported() else "float16 (GradScaler)","stage":"pretrain","steps":TRAIN_STEPS,"batch_size":BATCH_SIZE,"grad_accum":GRAD_ACCUM,"effective_batch":BATCH_SIZE*GRAD_ACCUM,"learning_rate":LR,"warmup_steps":WARMUP_STEPS,"output":str(OUTPUT_DIR),"resume":str(resume_path) if resume_path else None},ensure_ascii=False,indent=2))
print("========================")


## 10. Preflight

Checks model dimensions and exact parameter count, GPU, tokenizer, both splits, tiny forward/backward, AdamW optimizer, and output directory before full training. The tiny sanity model is only a check; training uses the configured Lyra 0.1 model.


In [ ]:
assert torch.cuda.is_available() and tokenizer_fingerprint and train_stats["examples"] > 0 and val_stats["examples"] > 0
assert cfg.vocab_size == tok.vocab_size and cfg.parameter_count() == parameter_count
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", CONFIG, "--require-cuda", "--instantiate", "--sanity"], cwd=PROJECT_DIR, check=True)
print("[OK] Python / PyTorch / CUDA / GPU / model config / tokenizer / train dataset / validation dataset / model initialization / forward / backward / optimizer / checkpoint directory")
print("READY TO TRAIN")


## 11. Start training

Uses the repository's existing `scripts/train.py`: AMP selects BF16 when supported and otherwise FP16 with GradScaler; checkpointing is every 100 steps with three rolling step snapshots. It trains on GPU 0.


In [ ]:
cmd = [sys.executable, "scripts/train.py", "--config", CONFIG, "--tokenizer", str(tokenizer_path), "--data", str(train_file), "--validation", str(val_file), "--stage", "pretrain", "--steps", str(TRAIN_STEPS), "--batch-size", str(BATCH_SIZE), "--grad-accum", str(GRAD_ACCUM), "--lr", str(LR), "--warmup-steps", str(WARMUP_STEPS), "--save-every", str(SAVE_EVERY), "--archive-every", str(SAVE_EVERY), "--keep-last-checkpoints", str(KEEP_LAST_N_CHECKPOINTS), "--eval-every", str(EVAL_EVERY), "--out", str(OUTPUT_DIR)]
if resume_path:
    cmd += ["--resume", str(resume_path)]
    if ALLOW_DATASET_CHANGE: cmd += ["--allow-dataset-change"]
print("Command:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## 12. Verify saved checkpoint

Fails loudly if the latest checkpoint cannot be loaded, its model/config state is inconsistent, or the tokenizer fingerprint differs.


In [ ]:
latest = OUTPUT_DIR / "latest.pt"
if not latest.is_file() or latest.stat().st_size <= 0: raise RuntimeError(f"Checkpoint missing or empty: {latest}")
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
if ckpt.get("config") != cfg.to_dict(): raise RuntimeError("Checkpoint config does not match current Lyra config")
if ckpt.get("metadata", {}).get("tokenizer_fingerprint") != tokenizer_fingerprint: raise RuntimeError("Checkpoint tokenizer fingerprint mismatch")
verify_model = LyraModel(cfg)
verify_model.load_state_dict(ckpt["model"], strict=True)
print({"checkpoint":"OK","bytes":latest.stat().st_size,"step":ckpt["step"],"stage":ckpt.get("stage"),"tokens_seen":ckpt["tokens_seen"],"best_validation_loss":ckpt.get("best_validation_loss"),"model_state":"strict load OK","config":"match","tokenizer_fingerprint":"match"})
del verify_model, ckpt; torch.cuda.empty_cache()


## 13. Resume in a new Kaggle session

After the run, use **Save Version ? Quick Save** (or **Save & Run All** only if you intentionally want a clean full rerun). Confirm the notebook output files are saved. In the next notebook session choose **Input ? Add Input ? Notebook Output Files**, attach the latest saved version, and run this notebook. `AUTO_RESUME=True` discovers `latest.pt`, restores model/optimizer/scheduler/scaler/RNG and step, and writes forward into a new version's `/kaggle/working`. Save a new version again to persist the next progress. Kaggle output is capped at 20 GB.

To intentionally expand/rebuild training data, attach or prepare the larger split, set `ALLOW_DATASET_CHANGE=True`, and resume. The trainer keeps optimizer/scheduler/step and validates model/tokenizer strictly. For SFT, set `RUN_NAME='stage3_sft'`, change the training stage in the command to `sft`, and add `--reset-stage` with a pretrained checkpoint; this resets the phase scheduler/step while preserving pretrained model weights.


## 14. Export model


In [ ]:
export_dir = Path("/kaggle/working/lyra_export"); export_dir.mkdir(parents=True, exist_ok=True)
ckpt = torch.load(OUTPUT_DIR / "latest.pt", map_location="cpu", weights_only=False)
torch.save(ckpt["model"], export_dir / "model.pt")
(export_dir / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2), encoding="utf-8")
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
readme = f"""Lyra 0.1 export
Model parameters: {parameter_count:,}
Vocabulary size: {cfg.vocab_size}
Context length: {cfg.context_length}
Training stage: {ckpt.get('stage', 'pretrain')}
Training step: {ckpt['step']}
Tokens seen: {ckpt['tokens_seen']}
Validation loss: {ckpt.get('best_validation_loss')}
Checkpoint source: {OUTPUT_DIR / 'latest.pt'}
Weights initialized from scratch; no pretrained model weights used.
"""
(export_dir / "README.txt").write_text(readme, encoding="utf-8")
print("Exported:", [str(p) for p in export_dir.iterdir()])
